# FTMO Decision-Tree Bot — Training + Learning Diagnostics (Colab)
Trains ONE tree, freezes it into a pure-Python evaluator, an MQL5 EA, and an RL alpha (slot 16),
then **measures whether it's actually learning** (sections 7–11).


## Step 0 — Upload the code (do this first)
Run the cell below. When the **Choose Files** button appears, pick the
**ftmo_dt_bot.zip** you downloaded. It unzips automatically.


In [ ]:
# pick ftmo_dt_bot.zip when prompted
from google.colab import files
import zipfile, io
up = files.upload()
name = [k for k in up if k.endswith('.zip')][0]
zipfile.ZipFile(io.BytesIO(up[name])).extractall('/content')
print('Done. Code is at /content/ftmo_dt_bot')


## 1. Code + data


In [ ]:
import sys, os
PKG_ROOT='/content/ftmo_dt_bot'   # folder you uploaded
sys.path.insert(0,PKG_ROOT)
os.makedirs('/content/data',exist_ok=True)


In [ ]:
import gdown
FILE_IDS={
    "EURUSD": "1tsR789vdRYE4rwDAE-hreWF2zN1slcjH",
    "GBPUSD": "1503qJQxjLwA2O0zIiakiOM_68Ucb-ypm",
    "XAUUSD": "1bzICq5oXh3z5PIwrovDa8xUHwRV-d8Gz",
    "US30": "1YF3vr4gBAm-4PZPLGCG0plWPDSaXH14h"
}
for sym,fid in FILE_IDS.items():
    out=f'/content/data/{sym}_M1.csv'
    if not os.path.exists(out): gdown.download(id=fid,output=out,quiet=False)
print(sorted(os.listdir('/content/data')))


## 2. Load + configure
Set `broker_utc_offset` to your FTMO server (≈2, 3 in DST). Account size is dynamic.


In [ ]:
from ftmo_dt.config import BotConfig
from ftmo_dt.data_loader import load_mt5_csv
cfg=BotConfig(account_balance=100_000, broker_utc_offset=2)
cfg.risk_per_trade_pct=0.0001   # 0.01%/trade
cfg.max_trades_per_day=800
frames={s:load_mt5_csv(f'/content/data/{s}_M1.csv',broker_utc_offset=cfg.broker_utc_offset) for s in FILE_IDS}
for s,d in frames.items(): print(s,d.shape,d.index.min(),'->',d.index.max())
# 800/day is a CAP, not a target -- it trades only on confident setups.
# To be MORE selective (fewer, cleaner trades for consistency):
#   from ftmo_dt.config import ftmo_consistent; cfg = ftmo_consistent(100_000)
#   or set: cfg.min_confidence = 0.55 ; cfg.deadband_cost_mult = 2.5


## 3. Train (per-symbol + pooled) and freeze every surface


In [ ]:
from ftmo_dt.train import train_from_frames
reports=train_from_frames(frames,cfg,out_dir='/content/out',backend='auto',pooled=True,per_symbol=True)
import json; print(json.dumps(reports.get('pooled',{}),indent=2,default=float))


## 4. Out-of-sample FTMO check + equity curves


In [ ]:
import pandas as pd, glob, matplotlib.pyplot as plt
for f in sorted(glob.glob('/content/out/reports/equity_pooled_*.csv')):
    s=pd.read_csv(f,index_col=0,parse_dates=True).iloc[:,0]
    plt.plot(s.index,s.values,label=f.split('_')[-1].replace('.csv',''))
plt.legend(); plt.title('Pooled tree — OOS equity'); plt.show()


In [ ]:
import json; sm=json.load(open('/content/out/reports/summary.json')); rows=[]
for model,by in sm.items():
    for sym,d in (by or {}).items():
        rows.append(dict(model=model,symbol=sym,ret=round(d['total_return'],4),maxDD=round(d['max_drawdown'],4),
                         worst_day=round(d['worst_day_pct'],4),trades=d['n_trades'],pf=round(d['profit_factor'],2),PASS=d['FTMO_PASS']))
import pandas as pd; display(pd.DataFrame(rows))


## 5. Ship the artifacts (regenerate EA + RL alpha from the trained tree)


In [ ]:
import numpy as np; from ftmo_dt.tree_model import TreeArrays; from ftmo_dt.feature_spec import FEATURES
from ftmo_dt.export_tree import write_mql5_ea, write_rl_alpha, write_register
d=np.load('/content/out/models/tree_pooled.npz',allow_pickle=True)
mc=float(d['min_confidence'][0]) if 'min_confidence' in d.files else 0.0
tree=TreeArrays(d['feature'],d['threshold'],d['children_left'],d['children_right'],d['value'],d['classes'],FEATURES,mc)
os.makedirs('/content/out/ea',exist_ok=True)
write_mql5_ea(tree,'/content/out/ea/FtmoDecisionTree.mq5',cfg,cfg.timeframes)
write_rl_alpha(tree,'/content/out/rl_alpha/decision_tree_ftmo_alpha.py',cfg.timeframes)
write_register('/content/out/rl_alpha/register_decision_tree_ftmo_alpha.py')
print('regenerated EA + RL alpha from the trained tree')


---
# Is it actually learning?
The model is **learning** when its OUT-OF-SAMPLE, after-cost edge is real and
consistent — not a lucky fit. Sections 7–11 measure that from five angles. Pick a symbol:


In [ ]:
from ftmo_dt import diagnostics as DG
SYM='EURUSD'   # <- symbol to diagnose


## 7. Learning scorecard
One call = the verdict. Reads: OOS profit factor (after costs), whether it beats a
**shuffled-label null** (the key test — if shuffled labels do as well, you're fitting noise),
walk-forward consistency, the train-vs-OOS overfit gap, and OOS directional hit-rate.


In [ ]:
report=DG.learning_scorecard(frames[SYM],cfg,SYM,backend='auto',n_folds=5)


## 7b. Signal-quality report
How well the +1/-1/0 SIGNAL itself does, out-of-sample vs in-sample. Watch `exp_bps`
(net-of-cost edge per fired signal) > 0 and a small in→out drop. See docs/TRAINING_AND_SIGNALS.md.


In [ ]:
DG.signal_report(frames[SYM], cfg, SYM, backend='auto')


## 8. Walk-forward (consistency across time)
Real edge shows up in MOST folds, not one. Bars above zero = profitable OOS windows.


In [ ]:
wf,agg=DG.walk_forward(frames[SYM],cfg,SYM,n_folds=6,backend='auto'); display(wf)
import matplotlib.pyplot as plt; plt.bar(wf['fold'],wf['ret']); plt.axhline(0,color='k',lw=.8)
plt.title(f'{SYM}: walk-forward OOS return per fold'); plt.ylabel('return'); plt.show()


## 9. Learning curve (does more data help?)
Rising/plateauing OOS PF above 1 = it's extracting signal. Flat at ~1 or falling = noise.


In [ ]:
lc=DG.learning_curve(frames[SYM],cfg,SYM,backend='auto'); display(lc)
plt.plot(lc['train_rows'],lc['oos_PF'],marker='o'); plt.axhline(1,color='r',ls='--')
plt.title(f'{SYM}: OOS profit factor vs training size'); plt.xlabel('train rows'); plt.ylabel('OOS PF'); plt.show()


## 10. Feature importance (what is it using?)
A sensible story (trend distance, CCI/RSI, volatility) is reassuring; one feature at ~100% is a red flag.


In [ ]:
display(DG.feature_importance(tree,top=15))


## 11. Parameter sweep (tune OOS, watch overfit)
Sort by `oos_PF`. If `train_PF` >> `oos_PF`, that row is overfit. At 800 trades/day,
raising `deadband_cost_mult` / `label_horizon_bars` trades fewer-but-cleaner setups.


In [ ]:
sweep=DG.param_sweep(frames[SYM],cfg,SYM,backend='auto'); display(sweep)


## 12. (Optional) save outputs to Drive
```python
from google.colab import drive; drive.mount('/content/drive')
!cp -r /content/out /content/drive/MyDrive/ftmo_dt_out
```
